# The GRPO Objective

Wiki reference for [the GRPO objective](https://ml-viz-ruby.vercel.app/wiki/grpo-objective).

**The idea in one sentence.** GRPO (used for RL fine-tuning of reasoning models) drops PPO's
value network: it samples a **group** of completions per prompt and uses their
**group-relative advantage** — standardize the rewards within the group — as the baseline,
combined with a PPO-style **clipped** update and a **KL penalty** to the reference model.

We implement the group-relative advantage and clipped objective from scratch, **validate the
standardized advantage and the clip**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Group-relative advantage

For a group of $G$ sampled completions with rewards $r_1,\dots,r_G$:

$$\hat A_i = \frac{r_i - \text{mean}(r)}{\text{std}(r) + \epsilon}$$

In [ ]:
def group_relative_advantage(rewards, eps=1e-8):
    rewards = np.asarray(rewards, dtype=float)
    mean, std = rewards.mean(), rewards.std()
    return (rewards - mean) / (std + eps)

# Four completions for one math prompt: verifier reward is 1 (correct) or 0 (incorrect)
rewards = np.array([1.0, 0.0, 1.0, 0.0])
advantages = group_relative_advantage(rewards)
for i, (r, a) in enumerate(zip(rewards, advantages), 1):
    print(f"completion {i}: reward={r:.1f}  advantage={a:+.2f}")

### Validate: the group-relative advantage is standardized

GRPO's baseline is the group mean, and it divides by the group std — so the advantages are
**zero-mean and unit-variance** within the group, and correct completions (reward 1) get a
**positive** advantage while incorrect ones go negative. We confirm.

In [ ]:
print('rewards   :', rewards)
print('advantages:', advantages.round(3))
assert np.isclose(advantages.mean(), 0, atol=1e-6), 'advantages are zero-mean (baseline = group mean)'
assert np.isclose(advantages.std(), 1, atol=1e-6), 'and unit-variance (standardized within the group)'
assert (advantages[rewards == 1] > 0).all(), 'correct completions get a positive advantage'
print('\n✅ no critic needed — the group itself provides the baseline')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#34d399' if r > 0 else '#f43f5e' for r in rewards]
axes[0].bar(range(1, 5), rewards, color=colors)
axes[0].set_title('Raw verifier reward'); axes[0].set_xlabel('completion'); axes[0].set_ylim(-1.5, 1.5)
axes[1].bar(range(1, 5), advantages, color=colors)
axes[1].axhline(0, color='white', lw=0.5)
axes[1].set_title('Group-relative advantage'); axes[1].set_xlabel('completion'); axes[1].set_ylim(-1.5, 1.5)
plt.tight_layout(); plt.show()
print("Group normalization turns flat 0/1 rewards into a signed push: up for correct, down for incorrect.")

## 2 — A harder prompt: all completions score the same

If every completion in the group gets the *same* reward, `std` is 0 and there is no learning signal — exactly the right behavior, since there is nothing to prefer between identical outcomes. The `eps` in the denominator only prevents a division-by-zero crash; it does not fabricate a signal.

In [ ]:
uniform_rewards = np.array([1.0, 1.0, 1.0, 1.0])
print('advantages when all completions agree:', group_relative_advantage(uniform_rewards))

## 3 — Clipped surrogate objective (per-token, PPO-style)

$$J(\theta) = \frac{1}{G}\sum_i \frac{1}{|o_i|}\sum_t \min\big(r_{i,t}\hat A_i,\ \text{clip}(r_{i,t}, 1-\epsilon, 1+\epsilon)\hat A_i\big)$$

We simulate token-level probability ratios $r_{i,t} = \pi_\theta / \pi_{\theta_{old}}$ directly (skipping an actual language model) to isolate how clipping behaves.

In [ ]:
def clipped_objective(ratios, advantage, clip_eps=0.2):
    unclipped = ratios * advantage
    clipped = np.clip(ratios, 1 - clip_eps, 1 + clip_eps) * advantage
    return np.minimum(unclipped, clipped)

ratios = np.linspace(0.4, 1.8, 200)
for adv, color, label in [(1.0, '#34d399', 'advantage = +1 (good completion)'),
                           (-1.0, '#f43f5e', 'advantage = -1 (bad completion)')]:
    obj = clipped_objective(ratios, adv)
    plt.plot(ratios, obj, color=color, label=label)
plt.axvline(1.0, color='white', lw=0.5, ls='--')
plt.xlabel('probability ratio r_t = pi_theta / pi_theta_old'); plt.ylabel('per-token objective')
plt.title('PPO-style clipping used inside GRPO'); plt.legend(); plt.tight_layout(); plt.show()
print("For a good completion the objective flattens once r_t > 1+eps: no reward for pushing the ratio further.")
print("For a bad completion it flattens once r_t < 1-eps: no extra penalty for already having suppressed it.")

### Validate: the clip caps the update

Like PPO, GRPO multiplies the advantage by the probability ratio but **clips** the ratio to
$[1-\epsilon, 1+\epsilon]$ and takes the min — so for a good completion (advantage $>0$) the
gain is capped at $(1+\epsilon)\cdot A$ no matter how large the ratio grows. We confirm the
cap.

In [ ]:
obj_good = clipped_objective(ratios, 1.0)
big_ratio_obj = obj_good[ratios > 1.5][0]
print(f'objective at ratio>1.5 (advantage +1): {big_ratio_obj:.3f}  (capped at 1+eps = 1.2)')
assert np.isclose(big_ratio_obj, 1.2, atol=1e-6), 'a good completion gains at most (1+eps)*A — the clip prevents runaway updates'
print('\n✅ clipping keeps each policy update in a trust region')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **flat groups** | all-same rewards → zero advantage → no signal (demo) |
| **no KL penalty** | the policy drifts / reward-hacks — anchor it to the reference |
| **group size** | too small = noisy baseline; larger = more compute per prompt |
| **verifier quality** | GRPO is only as good as the reward/verifier |
| **clip epsilon** | too large loses the trust region; too small stalls learning |

Demo: a group with no reward variance produces zero advantage.

In [ ]:
# GRPO's built-in failure mode: it needs REWARD VARIANCE within the group. If every completion
# gets the same verifier reward (all correct, or all wrong), the group std is ~0, so every
# advantage is 0 — there is NO gradient signal from that prompt. This is why GRPO benefits from
# prompts of the right difficulty (some right, some wrong). We confirm the dead group.
uniform = group_relative_advantage(np.array([1.0, 1.0, 1.0, 1.0]))
print(f'advantages when all 4 completions are correct: {uniform.round(3)}')
assert np.allclose(uniform, 0), 'no reward variance in the group -> zero advantage -> no learning signal'
print('\nToo-easy or too-hard prompts give a flat group -> no signal. Curriculum/difficulty matters for GRPO.')

## ✏️ Your turn

**Task — KL penalty term:** GRPO's full loss subtracts a KL penalty against a frozen reference policy:

$$\mathcal{L} = -J(\theta) + \beta \, D_{KL}(\pi_\theta \,\|\, \pi_{ref})$$

For discrete distributions, $D_{KL}(P\|Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)}$. Implement `kl_penalty(p, q)` for two probability vectors over the same token vocabulary, then check that it is exactly `0` when `p == q`, and positive otherwise.

In [ ]:
def kl_penalty(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    # TODO(you): return the KL divergence D_KL(p || q) = sum(p * log(p/q))
    return ...

policy = np.array([0.7, 0.2, 0.1])
reference = np.array([0.5, 0.3, 0.2])
same = np.array([0.7, 0.2, 0.1])

result = kl_penalty(policy, reference)
if result is not None:
    print('KL(policy || reference) =', result)
    print('KL(policy || policy)   =', kl_penalty(policy, same))
    assert kl_penalty(policy, same) < 1e-9
    assert result > 0
    print('Looks right: zero when distributions match, positive otherwise.')

<details><summary>Solution</summary>

```python
def kl_penalty(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(np.sum(p * np.log((p + eps) / (q + eps))))
```
The `eps` inside the log avoids `log(0)` if either distribution has a zero-probability token; it doesn't change the result when `p == q` since the ratio is still 1 for every nonzero entry.
</details>

## Key takeaways

- **No critic:** GRPO uses the **group mean** as the baseline — cheaper than PPO's value net.
- **Standardized advantage:** zero-mean, unit-variance within the group (verified).
- **Clipped update:** the gain is capped at $(1+\epsilon)A$ — a PPO-style trust region
  (verified).
- **Needs reward variance:** a uniform group gives no signal (demo) — plus a KL penalty keeps
  the policy near the reference.